# Notebook 0: 汇总各简单运动场景的最优参数集

扫描 `data/json/` 下所有贝叶斯优化结果，按运动类型分类，构建标准化参数字典。

**输出产物**: `artifacts/simple_params.pkl`
- `simple_params`: 简单运动 → SolverParams 字典
- `burpee_baseline_by_sample`: 波比跳样本 stem → 优化记录字典（P2 修复）

In [ ]:
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"D:\data\PPG_HeartRate\Algorithm\outline-PPGtoHR")
DATA_DIR = PROJECT_ROOT / "docs" / "research" / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "docs" / "research" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT / "python" / "src"))

from ppg_hr.params import SolverParams, fields as sp_fields

## 1. 运动场景定义与文件前缀映射

文件命名约定：`multi_{prefix}{N}.csv`，如 `multi_bobi1.csv`。
`PREFIX_MAP` 将中文拼音前缀映射为内部运动类别名。

In [ ]:
EXERCISES = {
    "jump_rope": {"prefix": "tiaosheng", "label": "跳绳", "complexity": "simple"},
    "arm_curl": {"prefix": "wanju", "label": "手臂弯举", "complexity": "simple"},
    "push_up": {"prefix": "fuwo", "label": "俯卧撑", "complexity": "simple"},
    "jumping_jack": {"prefix": "kaihe", "label": "开合跳", "complexity": "simple"},
    "burpee": {"prefix": "bobi", "label": "波比跳", "complexity": "complex"},
}
# 反向映射: 拼音前缀 -> 运动类别名
PREFIX_MAP = {info["prefix"]: name for name, info in EXERCISES.items()}

print("运动场景:")
for name, info in EXERCISES.items():
    tag = "[简单]" if info["complexity"] == "simple" else "[复杂]"
    print(f"  {tag} {info['label']} ({name}) -> prefix={info['prefix']}")

## 2. 扫描 JSON 优化结果

从 `data/json/` 读取所有 `*-best_params.json` 文件，按文件名提取运动类别和样本名。

In [ ]:
import re


def parse_json_filename(fp: Path) -> dict | None:
    """从 JSON 文件名解析运动类别、样本 stem、ppg_mode、filter.

    文件名格式: multi_{prefix}{N}-{ppg_mode}-{filter}-best_params.json
    """
    pattern = r"^(multi_[a-z]+(?:\d+))-(\w+)-(\w+)-best_params\.json$"
    m = re.match(pattern, fp.name)
    if not m:
        return None
    stem = m.group(1)  # e.g. "multi_bobi1"
    ppg_mode = m.group(2)
    filt = m.group(3)
    # 提取拼音前缀: multi_bobi1 -> bobi
    prefix_match = re.match(r"multi_([a-z]+)", stem)
    if not prefix_match:
        return None
    prefix = prefix_match.group(1)
    exercise = PREFIX_MAP.get(prefix)
    if exercise is None:
        return None
    return {
        "stem": stem,
        "exercise": exercise,
        "ppg_mode": ppg_mode,
        "adaptive_filter": filt,
    }


def scan_json_params(json_dir: Path) -> list[dict]:
    """扫描 json_dir 下所有 best_params 文件并解析."""
    results = []
    for fp in sorted(json_dir.glob("*-best_params.json")):
        parsed = parse_json_filename(fp)
        if parsed is None:
            print(f"  跳过无法解析的文件: {fp.name}")
            continue
        with open(fp, "r", encoding="utf-8") as f:
            data = json.load(f)
        results.append({**parsed, "source": str(fp), **data})
    return results


all_results = scan_json_params(DATA_DIR / "json")
print(f"共扫描到 {len(all_results)} 个优化结果:\n")
for r in all_results:
    label = EXERCISES[r["exercise"]]["label"]
    print(f"  {r['stem']}: {label}, min_err_hf={r.get('min_err_hf', float('nan')):.2f} BPM")

## 3. 优化效果对比

In [ ]:
rows = []
for r in all_results:
    info = EXERCISES[r["exercise"]]
    rows.append({
        "exercise": info["label"],
        "type": info["complexity"],
        "stem": r.get("stem", "?"),
        "ppg_mode": r.get("ppg_mode", "green"),
        "filter": r.get("adaptive_filter", "lms"),
        "min_err_hf": r.get("min_err_hf", float("nan")),
        "min_err_acc": r.get("min_err_acc", float("nan")),
    })

df_summary = pd.DataFrame(rows)
if not df_summary.empty:
    df_summary = df_summary.sort_values(["type", "exercise", "stem"])
    display(df_summary)
else:
    print("未找到任何优化结果!")

## 4. 构建标准化参数字典

P3 修复：`json_params_to_solver_params` 对 `SolverParams` 所有可调字段做全字段解码，
包括 KLMS/Volterra 策略专属参数。未知字段写入 `extras`。

In [ ]:
# P3: 全字段解码 — 自动映射 best_para_hf 中所有与 SolverParams 字段同名的键
_SOLVER_FIELD_NAMES = {f.name for f in sp_fields(SolverParams)} - {"extras"}


def json_params_to_solver_params(para_dict: dict, ppg_mode: str = "green", adaptive_filter: str = "lms") -> SolverParams:
    """将 JSON 中的 best_para_hf 字典转换为 SolverParams (全字段解码).

    与 para_dict 中键名匹配的 SolverParams 字段自动写入；
    未匹配的键存入 extras，避免丢失策略专属参数。
    """
    kwargs = {"ppg_mode": ppg_mode, "adaptive_filter": adaptive_filter}
    extras = {}
    for key, val in para_dict.items():
        if key in _SOLVER_FIELD_NAMES:
            kwargs[key] = val
        else:
            extras[key] = val
    if extras:
        kwargs["extras"] = extras
    return SolverParams(**kwargs)


# 按运动类别分组
from collections import defaultdict

results_by_exercise = defaultdict(list)
for r in all_results:
    results_by_exercise[r["exercise"]].append(r)

simple_params = {}
# P2: 按 stem 索引的波比跳基线字典
burpee_baseline_by_sample = {}

for name, info in EXERCISES.items():
    results = results_by_exercise.get(name, [])
    if not results:
        status = "!!! 缺失 !!!" if info["complexity"] == "simple" else "(评估目标, 可选)"
        print(f"  {info['label']} ({name}): {status}")
        continue

    if info["complexity"] == "simple":
        # 简单运动: 取 min_err_hf 最小的结果作为该类代表参数
        best = min(results, key=lambda r: r.get("min_err_hf", float("inf")))
        solver_p = json_params_to_solver_params(
            best["best_para_hf"],
            ppg_mode=best.get("ppg_mode", "green"),
            adaptive_filter=best.get("adaptive_filter", "lms"),
        )
        simple_params[name] = solver_p
        print(f"  {info['label']} ({name}): min_err_hf={best['min_err_hf']:.2f} BPM "
              f"[from {best['stem']}]")
    else:
        # 复杂运动: 每个 stem 独立记录 (P2)
        for r in results:
            burpee_baseline_by_sample[r["stem"]] = r
            print(f"  {info['label']} ({name}): stem={r['stem']}, "
                  f"min_err_hf={r['min_err_hf']:.2f} BPM [复杂场景基线]")

## 5. 可行性检查 & 保存

门控步骤: 确认所有简单场景参数就绪后保存。输出包含 P2 的按 stem 索引基线字典。

In [ ]:
missing = [EXERCISES[k]["label"] for k in EXERCISES
           if EXERCISES[k]["complexity"] == "simple" and k not in simple_params]

if missing:
    print("=" * 60)
    print("警告: 以下简单运动场景缺少优化结果:")
    for m in missing:
        print(f"  - {m}")
    print("\n请先对这些场景运行贝叶斯优化, 然后重新运行本 notebook.")
    print("=" * 60)
else:
    print(f"所有 {len(simple_params)} 个简单运动场景的参数已就绪:")
    for name, p in simple_params.items():
        print(f"  {name}: fs={p.fs_target}Hz, order={p.max_order}, smooth={p.smooth_win_len}")

    print(f"\n波比跳按样本基线 ({len(burpee_baseline_by_sample)} 个样本):")
    for stem, rec in burpee_baseline_by_sample.items():
        print(f"  {stem}: min_err_hf={rec['min_err_hf']:.2f} BPM")

    output = {
        "simple_params": simple_params,
        "burpee_baseline_by_sample": burpee_baseline_by_sample,
    }

    output_path = ARTIFACTS_DIR / "simple_params.pkl"
    with open(output_path, "wb") as f:
        pickle.dump(output, f)
    print(f"\n已保存到: {output_path}")